In [0]:
%pip install aiohttp
%pip install azure-eventhub
%restart_python


In [0]:
import asyncio
import aiohttp
from azure.eventhub.aio import EventHubProducerClient
from azure.eventhub import EventData
import json 
import requests
from databricks.sdk import WorkspaceClient

In [0]:
dbutils.widgets.text("scope_name", "")
SECRET_SCOPE = dbutils.widgets.get("scope_name")
dbutils.widgets.text("personal_conn_string_key", "")
personal_conn_string_key = dbutils.widgets.get("personal_conn_string_key")
dbutils.widgets.text("evh_name", "")
evh_name = dbutils.widgets.get("evh_name")

In [0]:
personal_conn_string = dbutils.secrets.get(scope=SECRET_SCOPE, key=personal_conn_string_key)

In [0]:
async def run(session, url):
    # Creating a producer client to send messages to the event hub.
    # Specifying a connection string to your event hubs namespace and
    # the event hub name.

    producer = EventHubProducerClient.from_connection_string(
        conn_str=personal_conn_string,
        eventhub_name=evh_name,
    )

    async with producer:
        # Creating a batch.
        batch = await producer.create_batch()
        last_sent_time = asyncio.get_event_loop().time()          

        async with session.get(url) as response:        
            async for line in response.content:  
                decoded_line = line.decode('utf-8').strip()   # line decoded to string                               

                try:
                    batch.add(EventData(decoded_line))
                except ValueError:                             # raises ValueError if max batch size has been exceeded
                    print(f"Number of elements in one batch: {len(batch)}")
                    await producer.send_batch(batch)
                    batch = await producer.create_batch()
                    batch.add(EventData(decoded_line))
                    last_sent_time = asyncio.get_event_loop().time() 
                
                if asyncio.get_event_loop().time() - last_sent_time > 5:          # if more than 5 seconds 
                    if len(batch) > 0:
                        print(f"Number of elements in one batch: {len(batch)}")
                        await producer.send_batch(batch)
                        batch = await producer.create_batch()
                        last_sent_time = asyncio.get_event_loop().time()


In [0]:
w = WorkspaceClient()
app_client_id = w.apps.get("api-app").oauth2_app_client_id

In [0]:
url = "https://adb-7405619910230738.18.azuredatabricks.net/oidc/v1/token"
notebook_token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

data = {
    "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
    "subject_token": notebook_token,
    "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
    "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
    "scope": "all-apis",
    "audience": app_client_id,
}

response = requests.post(url=url, data=data)
audience_token = response.json()["access_token"]

In [0]:
headers = {"Authorization": f"Bearer {audience_token}"}
custom_timeout = aiohttp.ClientTimeout(total=None)

In [0]:
URL = "https://api-app-7405619910230738.18.azure.databricksapps.com/api/stream" 
async with aiohttp.ClientSession(headers=headers, timeout = custom_timeout) as session:
    await run(session, URL) 